In [0]:
# --- 1. Parameter Configuration ---
catalog      = "sample_synthetic_sap"
catalog_path = "s3://databricks-sails-77e13b39-1a5a-444f-9be1-db63dc165aed/unity-catalog/4051546724732252/mock_sap_data"
schema_raw   = "sap"  # Where SAP tables live
schema_nodes = "silver"  # Where we build the Nodes
schema_edges = "gold"    # Where we build the Network Map

# Helper function to run SQL with parameters
def run_query(query_str):
    formatted_query = query_str.format(
        cat=catalog,
        raw=schema_raw,
        nodes=schema_nodes,
        edges=schema_edges
    )
    spark.sql(formatted_query)
    print(f"Executed successfully.")

# Initialize Schemas (Idempotent)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{catalog_path}'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_raw}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_nodes}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_edges}")

In [0]:
# --- 2. Create Silver Nodes (Entities) ---

# NODE: CUSTOMER
query_customer = """
CREATE OR REPLACE TABLE {cat}.{nodes}.node_customer AS
SELECT
  kunnr AS customer_id,
  name1 AS customer_name,
  ort01 AS city,
  land1 AS country,
  regio AS region
FROM {cat}.{raw}.kna1
"""

# NODE: MATERIAL (Joins MARA, MAKT, MBEW)
query_material = """
CREATE OR REPLACE TABLE {cat}.{nodes}.node_material AS
SELECT
  m.matnr AS material_id,
  t.maktx AS material_description,
  m.mtart AS material_type,
  m.meins AS base_uom,
  b.stprs AS standard_price,
  b.peinh AS price_unit
FROM {cat}.{raw}.mara m
LEFT JOIN {cat}.{raw}.makt t 
  ON m.matnr = t.matnr AND t.spras = 'E'
LEFT JOIN {cat}.{raw}.mbew b 
  ON m.matnr = b.matnr AND b.bwkey = '1000'
"""

# NODE: LOCATION (Unions Plant MARC and Storage MARD)
query_location = """
CREATE OR REPLACE TABLE {cat}.{nodes}.node_location AS
SELECT
  CONCAT(werks, '-', lgort) AS location_id,
  werks AS plant_id,
  lgort AS storage_loc_id,
  'Storage Location' AS type
FROM {cat}.{raw}.mard
UNION ALL
SELECT
  werks AS location_id,
  werks AS plant_id,
  NULL AS storage_loc_id,
  'Plant' AS type
FROM {cat}.{raw}.marc
"""

# Execute
run_query(query_customer)
run_query(query_material)
run_query(query_location)

In [0]:
# --- 3. Create Gold Edges (Relationships) ---

# EDGE: ORDERED_BY (Order -> Customer)
query_edge_order = """
CREATE OR REPLACE TABLE {cat}.{edges}.edge_ordered_by AS
SELECT
  vbak.vbeln AS source_order_id,
  vbak.kunnr AS target_customer_id,
  vbak.ERDAT AS order_date,
  vbak.vkorg AS sales_org,
  vbak.netwr AS total_value
FROM {cat}.{raw}.vbak
"""

# EDGE: INCLUDES_ITEM (Order -> Material)
query_edge_item = """
CREATE OR REPLACE TABLE {cat}.{edges}.edge_includes_item AS
SELECT
  vbap.vbeln AS source_order_id,
  vbap.matnr AS target_material_id,
  vbap.posnr AS item_number,
  vbap.kwmeng AS quantity_ordered,
  vbap.vrkme AS uom,
  vbap.werks AS shipping_plant_id
FROM {cat}.{raw}.vbap
"""

# EDGE: DOCUMENT_FLOW (Order -> Delivery)
# This links the process steps together using VBFA
# We map VBELN to Source (Order) and VBELN_N to Target (Delivery)
query_edge_flow = """
CREATE OR REPLACE TABLE {cat}.{edges}.edge_document_flow AS
SELECT
  vbfa.VBELN AS source_doc_id,      -- The Preceding Doc (Order)
  vbfa.VBELN_N AS target_doc_id,    -- The Subsequent Doc (Delivery)
  vbfa.VBTYP_V AS source_type,      -- 'C' for Order
  vbfa.VBTYP_N AS target_type,      -- 'J' for Delivery
  vbfa.RFMNG AS quantity_flowed,
  likp.lfdat AS delivery_date
FROM {cat}.{raw}.vbfa
LEFT JOIN {cat}.{raw}.likp 
  ON vbfa.VBELN_N = likp.vbeln      -- Join Delivery ID to LIKP
WHERE vbfa.VBTYP_V = 'C'            -- Filter for Order
  AND vbfa.VBTYP_N = 'J'            -- Filter for Delivery
"""

# EDGE: GOODS_MOVEMENT (Location -> Customer/Location)
# This uses MATDOC to track physical inventory moves
query_edge_move = """
CREATE OR REPLACE TABLE {cat}.{edges}.edge_goods_movement AS
SELECT
  m.MBLNR AS material_document_id,
  m.MATNR AS material_id,
  
  -- Source is always the Plant-StorageLoc where the document was posted
  CONCAT(m.WERKS, '-', m.LGORT) AS source_location_id,
  
  -- Logic to determine target
  CASE 
    -- Movement 601 = Goods Issue Delivery. The target is the Customer (KUNNR).
    WHEN m.BWART = '601' THEN m.KUNNR
    
    -- Movement 101 = Goods Receipt. The target is the Plant itself (Inbound).
    -- We map this as External -> Plant
    WHEN m.BWART = '101' THEN CONCAT(m.WERKS, '-', m.LGORT)

    -- For Transfers (301), we lack the 'Receiving Plant' column (UMWRK) in this dataset.
    -- We will label these generic 'Transfer_Out' for now.
    WHEN m.BWART IN ('301', '351') THEN 'Transfer_Out_Unknown_Dest'
    
    ELSE 'External_Usage' 
  END AS target_node_id,

  m.BWART AS movement_type,
  m.MENGE AS quantity_moved,
  m.MEINS AS uom,
  m.BUDAT AS posting_date
FROM {cat}.{raw}.matdoc m
WHERE m.BWART IN ('601', '641', '301', '101')
"""

# Execute
run_query(query_edge_order)
run_query(query_edge_item)
run_query(query_edge_flow)
run_query(query_edge_move)

In [0]:
%sql
-- Check the Gold layer to see if the network was built
SELECT * FROM mock_sap_data.gold.edge_document_flow LIMIT 10;